# Robo-Greeno Data A — Colab smoke test

5 cells. ~15 minutes total. Confirms the **MuJoCo + MuJoCo MJX + Stable-Baselines3** stack runs end-to-end on a free Colab T4 GPU.

Run this on Day 1 to verify your environment before starting the IK / IMU / RL labs.

**Before running:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save`.


## Step 0 — confirm GPU is allocated

Expected: a table showing **Tesla T4** with ~15 GB memory.

If it says "command not found" or "No GPU", the runtime didn't switch. Repeat the runtime-type change above and re-run this cell. Free quota is variable; if Colab refuses GPU, retry in 5 minutes or use a different Google account.


In [ ]:
!nvidia-smi | head -10


## Step 1 — install the stack (~2 min)

Installs MuJoCo (CPU + GL), MuJoCo MJX (JAX-based parallel sim), Stable-Baselines3 (RL), Gymnasium with MuJoCo envs, and mediapy (video display).


In [ ]:
!pip install -q mujoco mujoco-mjx stable-baselines3 gymnasium "gymnasium[mujoco]" mediapy
print("install OK")


## Step 2 — confirm MuJoCo CPU sim + render (~1 min)

Drops a red ball onto a grey plane and renders 120 frames offscreen. Proves MuJoCo CPU + offscreen rendering work — the same pipeline used for the IK and IMU labs.

Expected: a 2-second video of a bouncing red ball, plus `MuJoCo CPU sim + render: OK`.


In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")  # offscreen GL backend for headless Colab

import mujoco, mediapy as media, numpy as np

xml = """
<mujoco>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="2 2 0.1" rgba=".7 .7 .7 1"/>
    <body pos="0 0 .3">
      <joint type="free"/>
      <geom type="sphere" size=".1" rgba=".8 .2 .2 1"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=240, width=320)

frames = []
for _ in range(120):
    mujoco.mj_step(model, data)
    renderer.update_scene(data)
    frames.append(renderer.render())

media.show_video(frames, fps=60)
print("MuJoCo CPU sim + render: OK")


## Step 3 — confirm MJX runs on GPU (parallel sim) (~2 min)

Runs **1024 parallel sims** for 100 steps using MuJoCo MJX (the JAX rewrite). Proves MJX is using the GPU and that the Wk 6-8 parallel-RL phase will work on free Colab.

Expected:

```
JAX devices: [CudaDevice(id=0)]
100 steps × 1024 envs in 0.X s on gpu
```

If `JAX devices` shows `[CpuDevice(...)]`, JAX did not see the GPU — restart runtime (`Runtime → Restart`) and rerun cells 1-3.


In [ ]:
import jax, jax.numpy as jp
import mujoco
from mujoco import mjx
import time

print("JAX devices:", jax.devices())

xml = """
<mujoco>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="2 2 0.1"/>
    <body pos="0 0 .3">
      <joint type="free"/>
      <geom type="sphere" size=".1"/>
    </body>
  </worldbody>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(xml)
mx = mjx.put_model(m)
dx = mjx.make_data(mx)

# Batch of 1024 parallel sims
batched = jax.vmap(lambda _: dx)(jp.arange(1024))
step = jax.jit(jax.vmap(mjx.step, in_axes=(None, 0)))

# Warm-up the JIT
batched = step(mx, batched)
batched.qpos.block_until_ready()

t0 = time.time()
for _ in range(100):
    batched = step(mx, batched)
batched.qpos.block_until_ready()

elapsed = time.time() - t0
print(f"100 steps × 1024 envs in {elapsed:.2f}s on {jax.devices()[0].platform}")


## Step 4 — confirm SB3 training loop (~5 min)

Trains PPO on the MuJoCo `Ant-v4` environment for 20 000 steps. The reward will be poor — the policy is barely trained — but the point is the **training loop runs end-to-end with no errors**. Proves the SB3 + MuJoCo + Gymnasium plumbing works on free Colab.

Expected: a progress bar, then a final reward number (often between **−500 and +500**), and `SB3 training loop: OK`.


In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

env = gym.make("Ant-v4")          # MuJoCo ant — closest built-in env to a hexapod
model = PPO("MlpPolicy", env, verbose=0, device="cuda")

print("training PPO for 20k steps (~3-5 min)...")
model.learn(total_timesteps=20_000, progress_bar=True)

mean_reward, _ = evaluate_policy(model, env, n_eval_episodes=5)
print(f"mean reward after 20k steps: {mean_reward:.1f}")
print("SB3 training loop: OK")


## Success checklist

After all 5 cells run cleanly:

- [x] T4 GPU detected
- [x] MuJoCo simulates and renders a video
- [x] MJX runs 1024 parallel envs on GPU in well under 1 sec / 100 steps
- [x] SB3 trains a PPO policy without crashing

→ The entire Robo-Greeno Data A stack works on free Colab GPUs. No lab hardware required.

## Common pitfalls + fixes

| Symptom | Fix |
|---|---|
| `nvidia-smi: command not found` | Runtime → Change runtime type → T4 GPU → Save → rerun |
| MJX install errors on JAX version | Re-run Step 1 cell once (Colab caches partial installs) |
| `JAX devices: [CpuDevice...]` after install | Runtime → Restart runtime, rerun cells 1-3 |
| `mujoco.FatalError: gladLoadGL error` on render | Cell 2 already sets `MUJOCO_GL=egl` before importing mujoco |
| "No backend with GPU" on `gym.make` | SB3 falls back to CPU automatically — fine for confirmation |
| Colab disconnects mid-Step 4 training | Free tier has idle limits; reduce `total_timesteps` to 5_000 to confirm plumbing |

## Next steps after smoke test passes

- **Wk 1:** import a hexapod URDF (PhantomX, OpenPode) into MuJoCo and confirm it loads.
- **Wk 2-3:** swap the bouncing-ball XML for the hexapod and start the IK lab.
- **Wk 6-8:** scale Step 3 from 1024 to 4096+ parallel envs and wrap in a Brax/SBX-style PPO loop for the RL lab.
